# Weather Agent Demo

This notebook demonstrates the fixed weather agent with mocked API responses so the output is reproducible without external network access.

In [1]:
from pathlib import Path
import sys
from unittest.mock import patch

project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from graph import weather_agent

In [2]:
def mocked_get(url, params=None, timeout=None):
    class MockResponse:
        def __init__(self, data):
            self._data = data

        def json(self):
            return self._data

        def raise_for_status(self):
            return None

    if 'ipapi.co' in url:
        return MockResponse({
            'city': 'Test City',
            'region': 'Test Region',
            'country_name': 'Testland',
            'latitude': '10.0',
            'longitude': '20.0',
            'utc_offset': '+00:00',
            'timezone': 'UTC'
        })

    if 'open-meteo.com' in url:
        return MockResponse({
            'latitude': 10.0,
            'longitude': 20.0,
            'timezone': 'UTC',
            'utc_offset_seconds': 0,
            'current_weather_units': {
                'temperature': '°C',
                'windspeed': 'km/h',
                'winddirection': 'deg',
                'weathercode': 'code'
            },
            'current_weather': {
                'time': '2026-05-21T05:15:00Z',
                'temperature': 15.5,
                'windspeed': 5.0,
                'winddirection': 180,
                'is_day': 1,
                'weathercode': 0
            }
        })

    raise ValueError(f'Unhandled URL in notebook mock: {url}')

In [3]:
state = {
    'name': 'Tester',
    'location_data': None,
    'weather_data': None,
    'weather_info': None,
}

with patch('components.nodes.requests.get', side_effect=mocked_get):
    final_state = weather_agent.invoke(state)

print(final_state['weather_info'])

Time: 05:15 UTC | 05:15 (UTC+00:00)

Good morning, Tester!

Your current location: Test City, Test Region, Testland

Current weather conditions:
• Clear sky
• Temperature: 15.5°C (cool)
• Wind: 5.0 km/h
